In [ ]:
from private_config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
from torch.utils.data import DataLoader
from data_loader import UAVDataset, MetaDataset
from GPN_V4_adaMulti import GPN_Optimized, GPNTrainer
import torch
import gc
from draw_Grad_CAM import *

# 参数配置
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATASET_PATH = _cfg_require('Grad_CAM.ipynb.cell0.module.DATASET_PATH')
MODEL_PATHS = _cfg_require('Grad_CAM.ipynb.cell0.module.MODEL_PATHS')
SAVE_DIR = _cfg_require('Grad_CAM.ipynb.cell0.module.SAVE_DIR')
BLR=_cfg_require('Grad_CAM.ipynb.cell0.module.BLR')

N_WAY = _cfg_require('Grad_CAM.ipynb.cell0.module.N_WAY')
K_SHOT = _cfg_require('Grad_CAM.ipynb.cell0.module.K_SHOT')
Q_QUERY = _cfg_require('Grad_CAM.ipynb.cell0.module.Q_QUERY')  # 每个类<configured>个查询样本
SEED = _cfg_require('Grad_CAM.ipynb.cell0.module.SEED')
import torch
import numpy as np
import random
import os

# def set_global_seed(seed):
#     """
#     设置全局随机种子，以确保实验的可重现性。
#     """
#     # <configured>. Python 内置的 random 库
#     random.seed(seed)
#     # <configured>. NumPy 库
#     np.random.seed(seed)
# Historical configuration omitted; provide values through private configuration.
#     torch.manual_seed(seed)
# Historical configuration omitted; provide values through private configuration.
#     if torch.cuda.is_available():
#         # 为所有 GPU 设置相同的种子
#         torch.cuda.manual_seed_all(seed) 
#         torch.backends.cudnn.deterministic = True
#         torch.backends.cudnn.benchmark = False 

# Historical configuration omitted; provide values through private configuration.
#     os.environ['PYTHONHASHSEED'] = str(seed)

# # 调用函数以应用种子
# set_global_seed(SEED)
# print(f"全局随机种子已设置为: {SEED}")

# 加载数据
print("加载测试数据...")
test_dataset = UAVDataset(
    data_dir_path=DATASET_PATH,
    seed=SEED,
    is_train=False,
    train_ratio=_cfg_require('Grad_CAM.ipynb.cell0.module.train_ratio'),
    max_sample_count=_cfg_require('Grad_CAM.ipynb.cell0.module.max_sample_count'))

meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('Grad_CAM.ipynb.cell0.module.num_tasks'),seed=SEED)
test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('Grad_CAM.ipynb.cell0.module.batch_size'), shuffle=False)

model1 = GPN_Optimized(
             attention_type=_cfg_require('Grad_CAM.ipynb.cell0.module.attention_type'),
             reduction=_cfg_require('Grad_CAM.ipynb.cell0.module.reduction'))
trainer1 = GPNTrainer(model1,device=DEVICE,
                              mlr=BLR,use_Mdistance=_cfg_require('Grad_CAM.ipynb.cell0.module.use_Mdistance'),use_multi=_cfg_require('Grad_CAM.ipynb.cell0.module.use_multi'),save_middle=_cfg_require('Grad_CAM.ipynb.cell0.module.save_middle'),
                              prototypes_per_class=_cfg_require('Grad_CAM.ipynb.cell0.module.prototypes_per_class'),crop_ratio_h=_cfg_require('Grad_CAM.ipynb.cell0.module.crop_ratio_h'),crop_ratio_l=_cfg_require('Grad_CAM.ipynb.cell0.module.crop_ratio_l'),
                      resize=_cfg_require('Grad_CAM.ipynb.cell0.module.resize'),
                             merge_threshold=_cfg_require('Grad_CAM.ipynb.cell0.module.merge_threshold'), merge_start_epoch=_cfg_require('Grad_CAM.ipynb.cell0.module.merge_start_epoch'))
trainer1.load_model(MODEL_PATHS[0])
# 基线（无裁剪）
test_frequency_crop_statistics(
    trainer=trainer1,
    test_loader=test_loader,
    device=DEVICE,
    crop_ratio_h=_cfg_require('Grad_CAM.ipynb.cell0.module.crop_ratio_h__2'),
    crop_ratio_l=_cfg_require('Grad_CAM.ipynb.cell0.module.crop_ratio_l__2'),
    resize=_cfg_require('Grad_CAM.ipynb.cell0.module.resize__2'),
    save_dir=SAVE_DIR+_cfg_require('Grad_CAM.ipynb.cell0.module.path')
)


gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

model2 = GPN_Optimized(
             attention_type=_cfg_require('Grad_CAM.ipynb.cell0.module.attention_type__2'),
             reduction=_cfg_require('Grad_CAM.ipynb.cell0.module.reduction__2'))
trainer2 = GPNTrainer(model1,device=DEVICE,
                              mlr=BLR,use_Mdistance=_cfg_require('Grad_CAM.ipynb.cell0.module.use_Mdistance__2'),use_multi=_cfg_require('Grad_CAM.ipynb.cell0.module.use_multi__2'),save_middle=_cfg_require('Grad_CAM.ipynb.cell0.module.save_middle__2'),
                              prototypes_per_class=_cfg_require('Grad_CAM.ipynb.cell0.module.prototypes_per_class__2'),
                              crop_ratio_h=_cfg_require('Grad_CAM.ipynb.cell0.module.crop_ratio_h__3'),crop_ratio_l=_cfg_require('Grad_CAM.ipynb.cell0.module.crop_ratio_l__3'),resize=_cfg_require('Grad_CAM.ipynb.cell0.module.resize__3'),
                             merge_threshold=_cfg_require('Grad_CAM.ipynb.cell0.module.merge_threshold__2'), merge_start_epoch=_cfg_require('Grad_CAM.ipynb.cell0.module.merge_start_epoch__2'))
trainer2.load_model(MODEL_PATHS[1])
# 裁剪<configured>+<configured>并resize
test_frequency_crop_statistics(
    trainer=trainer2,
    test_loader=test_loader,
    device=DEVICE,
    crop_ratio_h=_cfg_require('Grad_CAM.ipynb.cell0.module.crop_ratio_h__4'),
    crop_ratio_l=_cfg_require('Grad_CAM.ipynb.cell0.module.crop_ratio_l__4'),
    resize=_cfg_require('Grad_CAM.ipynb.cell0.module.resize__4'),
    save_dir=SAVE_DIR+_cfg_require('Grad_CAM.ipynb.cell0.module.path__2')
)

# 裁剪<configured>+<configured>不resize
# test_frequency_crop_statistics(
#     trainer=trainer,
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
# )